# Face Attribute Classifier: Deep Learning & Benchmark Ensembles

Step-by-step tutorial classifying facial attributes on CelebA using four deep neural architectures:
- **ANN (Multi-Layer Perceptron)**: 2-layer dense baseline
- **Deeper Residual CNN**: 4-stage residual architecture with Batch Normalization and skip connections
- **RNN (Bi-LSTM)**: Spatial row-sequence modeling
- **Vision Transformer (ViT)**: Patch projection self-attention architecture

Integrated with MySQL relational storage (`celeba_db`) and benchmarked against 5-Fold Stratified Bagged XGBoost and Test-Time Augmentation (TTA).



In [ ]:
# Cell 1: Imports and setup
import os
import numpy as np
import pandas as pd
from PIL import Image
import pymysql
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)


## 1. Load Data from MySQL (or Local CSV)
Query the image file paths and binary labels from MySQL. If the database service is offline, read from the local CSV file instead.


In [ ]:
# Cell 2: Query records from MySQL with local CSV fallback
def load_dataset_records(sample_size=20000, target_attribute="Smiling"):
    try:
        # Connect to MySQL celeba_db on localhost
        conn = pymysql.connect(
            host="127.0.0.1",
            port=3306,
            user="root",
            password="",
            database="celeba_db",
            connect_timeout=3
        )
        query = f"""
        SELECT
            m.file_path,
            a.`{target_attribute}` AS label
        FROM images_metadata m
        INNER JOIN celeba_attributes a ON m.image_id = a.image_id
        LIMIT {sample_size};
        """
        df = pd.read_sql(query, conn)
        conn.close()
        print(f"Loaded {len(df)} rows from MySQL database.")
    except Exception:
        print("MySQL not reachable. Reading directly from list_attr_celeba.csv...")
        csv_path = "data/metadata/list_attr_celeba.csv"
        df = pd.read_csv(csv_path, nrows=sample_size)
        df["file_path"] = df["image_id"].apply(
            lambda name: os.path.join("img_align_celeba", name.replace(".jpg", ".png"))
        )
        df["label"] = df[target_attribute]

    # Convert CelebA -1 (No) and 1 (Yes) to 0 and 1
    df["label"] = (df["label"] == 1).astype(int)
    return df[["file_path", "label"]].copy()

df = load_dataset_records(sample_size=20000, target_attribute="Smiling")
df.head()


## 2. Image Preprocessing: Normalization and Standardization

Preprocessing prepares raw images for neural network training:
- **Normalization (Min-Max Scaling)**: Scales pixel values from [0, 255] to [0.0, 1.0].
- **Standardization (Z-Score Scaling)**: Centers pixel values to have zero mean and unit variance:
  $$Z = \frac{X - \mu_{\text{train}}}{\sigma_{\text{train}} + \epsilon}$$
  *Note: Statistics (mean and standard deviation) are computed strictly on the training set to prevent data leakage.*


In [ ]:
# Cell 3: Preprocessing with selectable Normalization & Standardization
IMG_SIZE = 64
CHANNELS = 3

def preprocess_images(data_df, scaling='standardize', test_size=0.15, val_size=0.15, random_state=42):
    images = []
    labels = []

    for _, row in data_df.iterrows():
        path = row['file_path']
        label = row['label']
        if os.path.exists(path):
            try:
                img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
                # 1. Normalization (Min-Max Scaling to [0.0, 1.0])
                norm_img = np.asarray(img, dtype=np.float32) / 255.0
                images.append(norm_img)
                labels.append(label)
            except Exception:
                continue

    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.float32)

    # Safe stratified split (70% Train, 15% Val, 15% Test)
    holdout = test_size + val_size
    unique, counts = np.unique(y, return_counts=True)
    strat_1 = y if (len(unique) > 1 and np.min(counts) >= 2) else None
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=holdout, random_state=random_state, stratify=strat_1)

    u_temp, c_temp = np.unique(y_temp, return_counts=True)
    strat_2 = y_temp if (len(u_temp) > 1 and np.min(c_temp) >= 2) else None
    val_frac = val_size / holdout
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=(1.0 - val_frac), random_state=random_state, stratify=strat_2)

    # 2. Standardization (Z-Score zero-mean and unit-variance)
    if scaling == 'standardize':
        # Compute statistics strictly from X_train to prevent data leakage
        train_mean = X_train.mean(axis=(0, 1, 2), keepdims=True)
        train_std = X_train.std(axis=(0, 1, 2), keepdims=True) + 1e-7
        X_train = (X_train - train_mean) / train_std
        X_val = (X_val - train_mean) / train_std
        X_test = (X_test - train_mean) / train_std
        print(f'Standardized -> Train Mean: {X_train.mean():.4f}, Train Std: {X_train.std():.4f}')
    else:
        print(f'Normalized -> Pixel Range: [{X_train.min():.1f}, {X_train.max():.1f}]')

    print(f'Split sizes: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}')
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

# Preprocess images with standardization (or set scaling="normalize")
(X_train, y_train), (X_val, y_val), (X_test, y_test) = preprocess_images(df, scaling='standardize')


## 3. Model 1: Artificial Neural Network (ANN)
Flattens the 64x64x3 pixel tensor into a 1D vector and uses dense layers as a baseline.


In [ ]:
# Cell 4: Build and train the ANN model
def build_ann():
    model = keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ], name="ann_classifier")

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

ann_model = build_ann()
ann_model.summary()

history_ann = ann_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, callbacks=[keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)],
    batch_size=32,
    verbose=1
)


## 4. Model 2: Convolutional Neural Network (CNN)
Uses 2D convolutions to learn spatial features like edges and facial landmarks.


In [ ]:
# Cell 5: Build and train the Deeper Multi-Block Residual CNN
def build_cnn():
    """
    Deeper Multi-Block Residual CNN:
    - 4 blocks: 32 -> 64 -> 128 -> 256 filters
    - BatchNormalization after each convolution
    - Residual skip connections (Add) to prevent vanishing gradients
    - GlobalAveragePooling2D for parameter efficiency
    """
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))

    # Block 1 (32 filters + residual)
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    sc1 = layers.Conv2D(32, (1, 1), padding='same')(inputs)
    x = layers.Add()([x, sc1])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 2 (64 filters + residual)
    sc2 = layers.Conv2D(64, (1, 1), padding='same')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, sc2])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 3 (128 filters + residual)
    sc3 = layers.Conv2D(128, (1, 1), padding='same')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, sc3])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 4 (256 filters)
    x = layers.Conv2D(256, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Classification head
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='residual_cnn_classifier')
    model.compile(optimizer=keras.optimizers.Adam(3e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_cnn()
cnn_model.summary()

history_cnn = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)],
    verbose=1
)


## 5. Model 3: Recurrent Neural Network (RNN / LSTM)
Processes each row of the image sequentially with an LSTM layer.


In [ ]:
# Cell 6: Build and train the RNN model
def build_rnn():
    model = keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS)),
        layers.Reshape((IMG_SIZE, IMG_SIZE * CHANNELS)),
        layers.LSTM(64),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ], name="rnn_classifier")

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

rnn_model = build_rnn()
rnn_model.summary()

history_rnn = rnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, callbacks=[keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)],
    batch_size=32,
    verbose=1
)


## 6. Model 4: Vision Transformer (ViT)
Projects 8x8 patches into token embeddings and applies multi-head self-attention.


In [ ]:
# Cell 7: Build and train the Vision Transformer model
def build_transformer():
    patch_size = 8
    num_patches = (IMG_SIZE // patch_size) ** 2  # 64 patches
    projection_dim = 64

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))

    # Project 8x8 patches into token embeddings
    projected = layers.Conv2D(projection_dim, (patch_size, patch_size), strides=patch_size, padding="valid")(inputs)
    patches = layers.Reshape((num_patches, projection_dim))(projected)

    # Self-attention block with residual connection
    attention = layers.MultiHeadAttention(num_heads=4, key_dim=projection_dim)(patches, patches)
    x = layers.Add()([attention, patches])

    # Global pooling and classification
    pooled = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(1, activation="sigmoid")(pooled)

    model = keras.Model(inputs=inputs, outputs=outputs, name="transformer_classifier")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

transformer_model = build_transformer()
transformer_model.summary()

history_transformer = transformer_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, callbacks=[keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)],
    batch_size=32,
    verbose=1
)


## 7. Model Evaluation and Comparison
Compare test accuracy, loss, precision, recall, and F1 score across all deep neural networks and evaluate against the 5-Fold Stratified Bagged XGBoost and TTA Ensemble benchmarks (which achieved **89.87%** test accuracy and **92.26%** precision).



In [ ]:
# Cell 8: Evaluate on unseen test set and plot curves
models = {
    "ANN": ann_model,
    "CNN": cnn_model,
    "RNN": rnn_model,
    "Vision Transformer": transformer_model
}

results = []
for name, model in models.items():
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    preds = (model.predict(X_test, verbose=0) > 0.5).astype(int)

    results.append({
        "Model": name,
        "Test Accuracy": f"{acc * 100:.2f}%",
        "Test Loss": round(loss, 4),
        "Precision": round(precision_score(y_test, preds, zero_division=0), 4),
        "Recall": round(recall_score(y_test, preds, zero_division=0), 4),
        "F1 Score": round(f1_score(y_test, preds, zero_division=0), 4)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Plot validation accuracy curves
if plt:
    plt.figure(figsize=(8, 4))
    plt.plot(history_ann.history['val_accuracy'], label="ANN", linestyle="--")
    plt.plot(history_cnn.history['val_accuracy'], label="CNN")
    plt.plot(history_rnn.history['val_accuracy'], label="RNN", linestyle=":")
    plt.plot(history_transformer.history['val_accuracy'], label="Vision Transformer")
    plt.title("Validation Accuracy Comparison")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
